# Wavefunction Viewer

Interactive viewer for wavefunctions stored in HDF5 files produced by the latom TDSE solver.
Use the slider to browse through real-time snapshots of $|\psi(x_1, x_2)|^2$.

In [ ]:
import sys
from pathlib import Path

import h5py
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

# Add scripts directory so we can import parser/plotting
scripts_dir = str(Path("../scripts").resolve())
if scripts_dir not in sys.path:
    sys.path.insert(0, scripts_dir)

from parser import list_hdf5_snapshots, load_wf_from_hdf5
from plotting import plot_wavefunction_2d

In [ ]:
# Path to the HDF5 file — adjust if your results are elsewhere
H5_PATH = Path("../res/wavefunctions.h5")

assert H5_PATH.exists(), f"HDF5 file not found: {H5_PATH}\nRun plotting.py first to collect wavefunctions."

In [ ]:
# Read grid metadata and list available snapshots
with h5py.File(str(H5_PATH), "r") as hf:
    nx = int(hf.attrs["nx"])
    ny = int(hf.attrs["ny"])
    real_dt = float(hf.attrs.get("real_dt", 0.1))

snapshots = list_hdf5_snapshots(H5_PATH)
print(f"Grid: {nx} x {ny}")
print(f"Found {len(snapshots)} wavefunction snapshots")
for s in snapshots[:5]:
    print(f"  {s['label']}")
if len(snapshots) > 5:
    print(f"  ... and {len(snapshots) - 5} more")

In [ ]:
# Read grid spacing from the config file if available
cfg_path = H5_PATH.parent / "simulation.cfg"
dx, dy = 0.2, 0.2  # defaults
if cfg_path.exists():
    for line in cfg_path.read_text().splitlines():
        line = line.strip()
        if line.startswith("grid_dx"):
            dx = float(line.split("=")[1].strip())
        elif line.startswith("grid_dy"):
            dy = float(line.split("=")[1].strip())
print(f"Grid spacing: dx={dx}, dy={dy} a.u.")

In [ ]:
# Build labels for the slider
labels = []
for s in snapshots:
    if "time_au" in s:
        labels.append(f"{s['label']} (t={s['time_au']:.1f} a.u.)")
    else:
        labels.append(s["label"])

# Interactive viewer
output = widgets.Output()

slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(snapshots) - 1,
    step=1,
    description="Snapshot:",
    continuous_update=False,
    layout=widgets.Layout(width="80%"),
)

label_display = widgets.HTML(value=f"<b>{labels[0]}</b>")

vmin_slider = widgets.IntSlider(
    value=7,
    min=1,
    max=15,
    step=1,
    description="Log orders:",
    continuous_update=False,
)


def update_plot(change=None):
    idx = slider.value
    label_display.value = f"<b>{labels[idx]}</b>"
    snap = snapshots[idx]
    wf = load_wf_from_hdf5(H5_PATH, group=snap["group"])

    with output:
        output.clear_output(wait=True)
        if wf is None:
            print(f"Failed to load {snap['group']}")
            return
        fig, ax = plt.subplots(figsize=(8, 7))
        plot_wavefunction_2d(ax, wf, dx, dy, nx, ny, vmin_orders=vmin_slider.value)
        ax.set_title(rf"$|\psi(x_1,x_2)|^2$ — {labels[idx]}")
        plt.tight_layout()
        plt.show()


slider.observe(update_plot, names="value")
vmin_slider.observe(update_plot, names="value")

display(widgets.VBox([slider, vmin_slider, label_display, output]))
update_plot()